In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import src.imports as lib

In [ ]:
# ==========================================
# 01 - DATA INTEGRATION
# ĐỌC 4 SHEET DỮ LIỆU LỚP 6–9
# ==========================================

file_path = lib.Path(
    "../data/data-raw/diem_THCS_10000_hoc_sinh_lop6_9_tuy_chon_THPT.xlsx"
)

sheet_names = ["Diem_L6", "Diem_L7", "Diem_L8", "Diem_L9"]

# Đọc dữ liệu Excel
dfs = lib.pd.read_excel(
    file_path,
    sheet_name=sheet_names,
    engine="openpyxl"
)

# Kiểm tra dữ liệu đã đọc
for sheet_name, df in dfs.items():
    print(f"{sheet_name}: {df.shape[0]:,} dòng × {df.shape[1]} cột")
    display(df.head())

In [ ]:
# ==========================================
# 01 - DATA INTEGRATION
# GHÉP DỮ LIỆU THEO ma_hoc_sinh
# ==========================================

id_col = "ma_hoc_sinh"

# Ghép 4 sheet theo mã học sinh
df_merged = dfs["Diem_L6"].merge(
    dfs["Diem_L7"],
    on=id_col,
    how="inner",
    suffixes=("_L6", "_L7"),
    validate="one_to_one"
)

df_merged = df_merged.merge(
    dfs["Diem_L8"],
    on=id_col,
    how="inner",
    suffixes=("", "_L8"),
    validate="one_to_one"
)

df_merged = df_merged.merge(
    dfs["Diem_L9"],
    on=id_col,
    how="inner",
    suffixes=("", "_L9"),
    validate="one_to_one"
)

# Kiểm tra kết quả
print(f"Số dòng sau ghép: {df_merged.shape[0]:,}")
print(f"Số cột sau ghép: {df_merged.shape[1]}")
print(f"Số mã học sinh duy nhất: {df_merged[id_col].nunique():,}")

display(df_merged.head())

In [ ]:
# ==========================================
# 01 - DATA INTEGRATION
# ĐỔI TÊN CỘT ĐIỂM THEO TIỀN TỐ LỚP 6–9
# ==========================================

id_col = "ma_hoc_sinh"

# Đổi tên cột điểm trong từng sheet
for grade in [6, 7, 8, 9]:
    sheet_name = f"Diem_L{grade}"

    dfs[sheet_name] = dfs[sheet_name].rename(
        columns={
            col: f"L{grade}_{col}"
            for col in dfs[sheet_name].columns
            if col.endswith(("_hki", "_hkii", "_ca_nam"))
        }
    )

# Ghép lại dữ liệu sau khi đổi tên
df_merged = dfs["Diem_L6"].copy()

for grade in [7, 8, 9]:
    df_merged = df_merged.merge(
        dfs[f"Diem_L{grade}"],
        on=id_col,
        how="inner",
        suffixes=("", f"_L{grade}"),
        validate="one_to_one"
    )

# Kiểm tra
print(f"Kích thước dữ liệu: {df_merged.shape}")
display(df_merged.head())
print(df_merged.columns.tolist())

In [ ]:
# ==========================================
# 01 - DATA INTEGRATION
# CHỈ GIỮ MỘT BỘ 9 NHÃN LỰA CHỌN MÔN HỌC
# ==========================================

choice_cols = [
    "Vật lí", "Hóa học", "Sinh học", "Địa lí",
    "Giáo dục kinh tế và pháp luật (GDKT&PL)",
    "Tin học", "Công nghệ", "Âm nhạc", "Mĩ thuật"
]

# Giữ bộ nhãn từ lớp 9, loại các bộ nhãn còn lại
cols_to_drop = [
    f"{col}_L{grade}"
    for grade in [7, 8, 9]
    for col in choice_cols
    if f"{col}_L{grade}" in df_merged.columns
]

df_merged = df_merged.drop(columns=cols_to_drop)

# Kiểm tra kết quả
print(f"Kích thước dữ liệu: {df_merged.shape}")
print(f"Số cột nhãn: {sum(col in df_merged.columns for col in choice_cols)}")

display(df_merged.head())

In [ ]:
# ==========================================
# 01 - DATA INTEGRATION
# KIỂM TRA MỖI HỌC SINH CHỈ XUẤT HIỆN MỘT LẦN
# ==========================================

id_col = "ma_hoc_sinh"

# Kiểm tra mã học sinh trùng lặp
duplicate_mask = df_merged[id_col].duplicated(keep=False)

duplicate_count = df_merged[id_col].duplicated().sum()
missing_count = df_merged[id_col].isna().sum()
unique_count = df_merged[id_col].nunique()

print("=" * 60)
print("KIỂM TRA TÍNH DUY NHẤT CỦA MÃ HỌC SINH")
print("=" * 60)

print(f"Tổng số dòng: {len(df_merged):,}")
print(f"Số mã học sinh duy nhất: {unique_count:,}")
print(f"Số mã học sinh bị thiếu: {missing_count:,}")
print(f"Số dòng trùng mã (không tính lần đầu): {duplicate_count:,}")

# Đánh giá kết quả
if duplicate_count == 0 and missing_count == 0:
    print("\nPASS: Mỗi học sinh chỉ xuất hiện một lần.")
else:
    print("\nFAIL: Phát hiện mã học sinh trùng hoặc bị thiếu.")

    # Hiển thị các dòng trùng mã
    if duplicate_count > 0:
        display(
            df_merged.loc[duplicate_mask]
            .sort_values(id_col)
        )

In [ ]:
# ==========================================
# 01 - DATA INTEGRATION
# XÁC NHẬN DỮ LIỆU CÓ 10.000 DÒNG VÀ 106 CỘT
# ==========================================

expected_rows = 10000
expected_cols = 106

actual_rows, actual_cols = df_merged.shape

print("=" * 60)
print("KIỂM TRA KÍCH THƯỚC DỮ LIỆU SAU GHÉP")
print("=" * 60)

print(f"Số dòng thực tế: {actual_rows:,} / {expected_rows:,}")
print(f"Số cột thực tế: {actual_cols} / {expected_cols}")

if (actual_rows, actual_cols) == (expected_rows, expected_cols):
    print("\nPASS: Dữ liệu có đúng 10.000 dòng và 106 cột.")
else:
    print("\nFAIL: Kích thước dữ liệu không đúng yêu cầu.")

# Kiểm tra chi tiết cấu trúc
score_cols = [
    col for col in df_merged.columns
    if col.startswith(("L6_", "L7_", "L8_", "L9_"))
    and col.endswith(("_hki", "_hkii", "_ca_nam"))
]

print(f"\nSố cột điểm: {len(score_cols)} / 96")
print(f"Số cột nhãn: {sum(col in df_merged.columns for col in choice_cols)} / 9")
print(f"Số cột định danh: {int('ma_hoc_sinh' in df_merged.columns)} / 1")

assert df_merged.shape == (10000, 106), "Kích thước dữ liệu không hợp lệ!"
assert len(score_cols) == 96, "Số cột điểm không hợp lệ!"
assert sum(col in df_merged.columns for col in choice_cols) == 9, "Số cột nhãn không hợp lệ!"

print("\nPASS: Kiểm tra cấu trúc dữ liệu thành công.")

In [ ]:
# ==========================================
# 01 - DATA INTEGRATION
# ĐƯA 9 CỘT NHÃN VỀ CUỐI VÀ LƯU DỮ LIỆU
# ==========================================

choice_cols = [
    "Vật lí", "Hóa học", "Sinh học", "Địa lí",
    "Giáo dục kinh tế và pháp luật (GDKT&PL)",
    "Tin học", "Công nghệ", "Âm nhạc", "Mĩ thuật"
]

# Sắp xếp: Mã học sinh → Điểm lớp 6–9 → 9 nhãn
score_cols = [
    col for col in df_merged.columns
    if col.startswith(("L6_", "L7_", "L8_", "L9_"))
    and col.endswith(("_hki", "_hkii", "_ca_nam"))
]

df_merged = df_merged[
    ["ma_hoc_sinh"] + score_cols + choice_cols
]

# Kiểm tra cấu trúc
assert df_merged.shape == (10000, 106)

print("9 cột cuối cùng:")
print(df_merged.columns[-9:].tolist())

display(df_merged.head())

# Lưu dữ liệu
output_dir = lib.Path("../data/data-clean")
output_dir.mkdir(parents=True, exist_ok=True)

output_file = output_dir / "hoc_sinh_lop6_9_clean.xlsx"

df_merged.to_excel(
    output_file,
    index=False,
    engine="openpyxl"
)

print(f"Đã lưu dữ liệu: {output_file.resolve()}")
print(f"Kích thước: {df_merged.shape[0]:,} dòng × {df_merged.shape[1]} cột")